# ❄️ Frozen Lake – A Gentle Introduction to Reinforcement Learning

<img src="img/frozen_lake.png" alt="Frozen Lake Problem" width="400"/>

Welcome to the Frozen Lake!

In this notebook we'll use a tiny grid-world game to get an **intuitive** feel for some core ideas of **Reinforcement Learning (RL)**:

- An **agent** (our little elf) moves on a grid.
- The grid is the **environment** (the frozen lake).
- At each step, the agent chooses an **action** (left, down, right, up).
- The environment responds with a new **state** (where we are) and a **reward** (good or bad).
- Our goal is to understand how **good** it is to be in each state if we follow a certain **strategy**.

We'll start by playing the game ourselves, then let simple strategies control the agent, and finally estimate how good each state is under a given strategy.

### The building blocks of every RL problem

| Concept | Symbol | Meaning in Frozen Lake |
|---|---|---|
| **Environment** | — | The frozen lake grid |
| **State** | $s$ | Which tile the elf is on (0–63 for an 8×8 map) |
| **Action** | $a$ | Left (0), Down (1), Right (2), Up (3) |
| **Reward** | $r$ | +1 at goal, −1 in a hole, 0 otherwise |
| **Policy** | $\pi(a \mid s)$ | The strategy: probability of action $a$ in state $s$ |
| **Transition** | $p(s' \mid s, a)$ | Where the elf ends up (stochastic on slippery ice) |

<div align="center">
  <img src="../lab2/img/rl-loop.png" alt="RL interaction loop" width="600"/>
  <p><em>Figure: The reinforcement-learning interaction loop.</em></p>
</div>

---

## 🗺️ Today's Journey

| Section | What you'll see | Key idea |
|--------|------------------|----------|
| **§0 – Setup** | Register the environment | Gymnasium API |
| **§1 – Meet the Environment** | You play Frozen Lake with the arrow keys | States, actions, rewards |
| **§2 – Strategies (Policies)** | Simple strategies that choose actions automatically | $\pi(a \mid s)$ is a rule for picking actions |
| **§3 – Monte Carlo Rollouts** | Estimate how good each state is by averaging many episodes | Learn from complete returns $G_t$ |
| **§4 – TD(0)** | Estimate values step by step | Bootstrapping with the Bellman equation |
| **Bonus – Slippery World** | Turn on stochastic transitions | The same action can lead to different places |
| **§5 – Q-Learning** | Learn a *good* strategy automatically | Off-policy control with ε-greedy |

> 💡 **Big picture:** we're not trying to build the smartest agent possible.  
> We're using Frozen Lake as a **playground** to see how strategies and "state values" behave in a small, visual world.


In [ ]:
import gymnasium as gym
import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)

## 🧩 §0. Registering the Frozen Lake Environment

Before we can interact with Frozen Lake, we register our (slightly customized) version with Gymnasium. A few parameters worth noting:

- `map_name="8x8"` – the grid size (64 states).
- `is_slippery` – if `True`, actions sometimes slip to a neighboring direction (stochastic transitions, see the Bonus section).
- `success_rate=3/4` – when slippery, the intended action happens 75% of the time.
- `reward_schedule=(1, -1, 0)` – reward for reaching the goal, falling in a hole, and every other step.
- `max_episode_steps=200` – episodes are cut off after 200 steps to avoid endless wandering.


In [ ]:
gym.register(
    id="FrozenLake-v2",
    entry_point="techdays26.frozen_lake.frozen_lake_enhanced:FrozenLakeEnv",
    kwargs={"map_name": "8x8"},
    max_episode_steps=200,
    reward_threshold=0.85,
)


def make_frozen_lake(render_mode="human", show_values=False, slippery=False):
    """Create a Frozen Lake env with our default parameters.

    - show_values=True overlays V(s) (or Q(s, a)) on the grid.
    - slippery=True turns on stochastic transitions (see the Bonus section).
    """
    return gym.make(
        "FrozenLake-v2",
        desc=None,  # pass a custom map here, or use map_name
        map_name="8x8",
        show_q_labels=show_values,
        is_slippery=slippery,
        success_rate=3
        / 4,  # probability the intended action happens (only if slippery)
        reward_schedule=(1, -1, 0),  # (goal, hole, every other step)
        render_mode=render_mode,
    )

## 🧊 1. Meet the Environment: Play Frozen Lake Yourself

Before we talk about strategies and value functions, let’s **experience** the environment.

- Use the **arrow keys** to move the elf.
- Try to reach the **goal** tile (G).
- Avoid the **holes** (H) – if you fall in, the episode ends.
- The reward is:
  - **+1** for reaching the goal
  - **−1** for falling into a hole
  - **0** for all other moves

In this first part, *you* are the strategy.  
Later, we’ll replace you with a function that chooses actions automatically.

In [ ]:
# ▶ Tip: keep `episodes` small — you'll be pressing arrow keys!
#    The Pygame window must be focused for key presses to register. ESC quits.
episodes: int = 2

env = make_frozen_lake()

try:
    env.unwrapped.set_info({"Mode": "Manual (Arrow Keys)"})

    # ╭──────────────────────────────────────────────────────────────╮
    # │ ★ CORE PATTERN — the Gymnasium interaction loop              │
    # │     state = env.reset()                                      │
    # │     while not done:                                          │
    # │         action = <pick somehow>                              │
    # │         state, reward, term, trunc, info = env.step(action)  │
    # │ Every RL algorithm in this notebook wraps this exact loop.   │
    # ╰──────────────────────────────────────────────────────────────╯
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)  # overlay episode number on the UI

            # ← YOU are the policy here: keyboard input → action
            action = get_action_from_keyboard(key_to_action)
            if action is None:
                raise SystemExit  # ESC pressed

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

        # Brief pause after episode ends so you can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🧭 2. Strategies (Policies): $\pi(a \mid s)$ in Plain Language

A **policy** (we’ll write it as $\pi$) is just a **strategy**:

> Given the current state, how likely are we to choose each possible action?

In code, we represent a policy as a function:

```python
def π(state, action) -> float:
    """Return the probability of taking `action` in `state`."""
```

To actually **pick** an action, we:

1. Ask the policy for the probability of each action.
2. Randomly choose one action according to these probabilities.

```python
def act(state, π):
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)
```

So:

- `π` describes the **behavior** (“go down 45% of the time, right 45%, …”).
- `act` uses that behavior to pick a concrete action at each step.

### Example Strategies

Below we define two simple strategies:

**Question(s) for you:**  
- What do the following policies do? Can you guess?
- Before running the code, try to imagine what the elf will do with each strategy. Will it reach the goal? Will it get stuck?

In [ ]:
from techdays26.frozen_lake.frozen_lake_utils import action_to_arrow

print("Mapping of actions to directions:")
print(action_to_arrow)

In [ ]:
# ╭──────────────────────────────────────────────────────────────╮
# │ ★ CORE IDEA — a policy π(a | s) returns a PROBABILITY.       │
# │   To actually pick an action, we SAMPLE from that            │
# │   distribution with `act()`.                                 │
# ╰──────────────────────────────────────────────────────────────╯
def act(state: int, π) -> int:
    probs = [π(state, a) for a in range(4)]  # evaluate π(a|s) for every action
    return np.random.choice([0, 1, 2, 3], p=probs)  # sample one action


def π_1(state: int, action: int) -> float:
    """π(a|s) → probability. Biased random walk: mostly Down / Right."""
    #         ←    ↓    →    ↑
    probs = [0.05, 0.45, 0.45, 0.05]
    return probs[action]


def π_2(state: int, action: int) -> float:
    """π(a|s) → probability. Deterministic: Right on every row, Down in the last column (8×8 grid)."""
    if state % 8 < 7:
        #         ←    ↓    →    ↑
        probs = [0.0, 0.0, 1.0, 0.0]
    else:
        #         ←    ↓    →    ↑
        probs = [0.0, 1.0, 0.0, 0.0]
    return probs[action]


class EpsilonGreedyPolicy:
    """ε-greedy policy derived from a Q-table. Used later in §5 (Q-learning).

    With probability 1-ε pick a greedy action (argmax over Q), otherwise pick
    a uniform random action:

        π_q(a|s) = (1-ε)/|A*| for greedy actions a ∈ A*  +  ε/|A| for all actions,
    where A* = argmax_a Q(s, a).
    """

    def __init__(self, q: np.ndarray, epsilon: float, epsilon_decay: float = 0.0):
        self.q = q  # stored by reference: updates to `q` are seen here
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay

    def __call__(self, state: int, action: int) -> float:
        nA = self.q.shape[1]

        # Random part
        random_prob = self.epsilon / nA

        # Greedy part
        max_q = np.max(self.q[state, :])
        greedy_actions = np.where(np.abs(self.q[state, :] - max_q) < 1e-6)[0]
        greedy_prob_each = (1.0 - self.epsilon) / len(greedy_actions)

        if action in greedy_actions:
            return random_prob + greedy_prob_each
        else:
            return random_prob

    def decay(self):
        """Shrink ε after each episode (exploration → exploitation)."""
        self.epsilon = max(self.epsilon - self.epsilon_decay, 0.0)

### 👀 2.1 Watching a Strategy in Action

In the next cell, you can choose:

- `π = π_1` or `π = π_2` for an **automatic** strategy, or
- `π = "manual"` to control the agent yourself with the arrow keys.

While it runs, pay attention to:

- The **path** the elf tends to take.
- How often it reaches the goal vs. falls into holes.
- How “smart” the behavior looks, given how simple the strategy is.

In [ ]:
episodes: int = 20
π = "manual"  # Callable or "manual"
# π = π_1
# π = π_2

env = make_frozen_lake()

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            # ── ★ The ONLY difference from §1: where the action comes from ──
            if π == "manual":
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)  # ← sample from π(·|s)
            # ────────────────────────────────────────────────────────────────

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🎲 §3. Estimating the Value Function $V^\pi(s)$ with Monte Carlo Rollouts

We now fix a strategy $\pi$ (say π₁ or π₂) and ask:

> If we start in state $s$ and follow $\pi$ until the episode ends,  
> **how much total reward do we expect to collect?**

### 📐 The return $G_t$ and the value $V^\pi(s)$

The **return** at time $t$ is the (discounted) sum of rewards from then until the end of the episode:

$$
G_t \;=\; R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots \;=\; \sum_{k=0}^{\infty} \gamma^k \, R_{t+k+1}
$$

The **state-value function** under policy $\pi$ is the *expected* return when we start in state $s$:

$$
\boxed{\; V^\pi(s) \;=\; \mathbb{E}_\pi \!\left[\, G_t \,\big|\, S_t = s \,\right] \;}
$$

Why the expectation? Because the environment (and possibly $\pi$) are random: $G_t$ differs each episode. We care about its **average**.

The discount $\gamma \in [0, 1)$ controls how much future vs. immediate rewards matter:
- $\gamma = 0$ → only the next reward matters.
- $\gamma$ near 1 → rewards far in the future still matter almost as much.
- It also keeps the sum finite for long or infinite episodes.

### 💡 §3.1 Monte Carlo: estimate the expectation by sampling

We can't compute $\mathbb{E}_\pi[G_t \mid S_t = s]$ in closed form (we don't have the full model). So we **sample** it:

1. Pick a strategy $\pi$.
2. Run an episode and store the trajectory $(s_0, r_1, s_1, r_2, \dots)$.
3. For each state visited, compute the return $G_t$ from that time onward.
4. Repeat many episodes and **average** the returns for each state:

$$
V^\pi(s) \;\approx\; \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_t^{(i)} \qquad \text{(average over visits to }s\text{)}
$$

This is **Monte Carlo** (estimate by random sampling, as in a casino) using **rollouts** (full episodes). We'll use the **first-visit** variant: within one episode we only count the *first* time each state is visited. This keeps the $N(s)$ samples independent and the estimator unbiased.

> 👀 **What you'll see:** while episodes roll out, $V^\pi(s)$ is drawn on each tile and updates live as more episodes accumulate. Unvisited states stay at 0.


In [ ]:
# ▶ Tip: use π_1 or π_2 (not manual) for the large episode count below.
#    If you set π = "manual", reduce `episodes` to something you'd actually play.
episodes: int = 20000

π = π_1  # Callable or "manual"
# π = π_2
# π = "manual"

# Deterministic world for now; we'll enable slippery=True in the Bonus section.
env = make_frozen_lake(show_values=True, slippery=False)

################################################################################
nS = env.observation_space.n
v = np.zeros((nS,))  # our current estimate of V^π(s)
returns_sum = np.zeros(nS)  # running Σ of G_t seen at each state
returns_count = np.zeros(nS)  # N(s): number of first-visits to s so far
gamma = 0.9  # discount factor γ
################################################################################

try:
    for i in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False

        # 📝 Step A — record the full trajectory of this episode (the "rollout")
        episode = []

        while not terminated and not truncated:
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            if π == "manual":
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            episode.append((state, reward))  # ← collect (s_t, r_{t+1})

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # ╭──────────────────────────────────────────────────────────────╮
        # │ ★ CORE IDEA — First-visit Monte Carlo update                 │
        # │   Walk the trajectory BACKWARDS, computing                   │
        # │       G_t = r_{t+1} + γ · G_{t+1}                            │
        # │   For the FIRST time we visit each state in this episode,    │
        # │   push V(s) toward the running mean of observed returns:     │
        # │       V(s) ← (1 / N(s)) · Σ G_t^(i)                          │
        # ╰──────────────────────────────────────────────────────────────╯
        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = r_t + gamma * G  # ← G_t computed backwards
            if s_t not in visited:  # ← first-visit filter
                visited.add(s_t)
                returns_sum[s_t] += G
                returns_count[s_t] += 1
                v[s_t] = returns_sum[s_t] / returns_count[s_t]  # ← running mean

        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

In [ ]:
import matplotlib.pyplot as plt

# Assume `v` already contains your learned value function (shape: n_states)

# 1. Create an env in rgb_array mode
env = make_frozen_lake(render_mode="rgb_array", show_values=True)

# 2. Reset and set the value function for rendering
state, _ = env.reset()
env.unwrapped.set_v(v)
env.unwrapped.set_episode("final")
env.unwrapped.set_info({"Note": "Final value function V^π(s)"})

# 3. Render one frame as an RGB array
rgb = env.render()  # shape: (H, W, 3), dtype=uint8

env.close()

# 4. Plot with matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.axis("off")
plt.title("Final value function $V^π(s)$ for policy using MC Learning")
plt.show()

## 🔁 §4. TD(0): Learning $V^\pi(s)$ Step by Step

Monte Carlo waits until **the end of an episode** to update $V$. What if we want to learn **after every single step**? That's **TD(0)**.

### 📐 From the Bellman equation to the TD(0) update

Start again from the value definition, but split the return into *next reward* + *rest*:

$$
V^\pi(s) \;=\; \mathbb{E}_\pi[\,G_t \mid S_t = s\,]
\;=\; \mathbb{E}_\pi[\,R_{t+1} + \gamma\, G_{t+1} \mid S_t = s\,]
$$

Because $\mathbb{E}_\pi[\,G_{t+1} \mid S_{t+1} = s'\,] = V^\pi(s')$, we can replace $G_{t+1}$ by $V^\pi(S_{t+1})$. That gives the **Bellman expectation equation** for $V^\pi$:

$$
\boxed{\; V^\pi(s) \;=\; \mathbb{E}_\pi\!\left[\, R_{t+1} + \gamma\, V^\pi(S_{t+1}) \,\big|\, S_t = s \,\right] \;}
$$

TD(0) turns this equation into a **learning rule** by using one observed step as a noisy sample of the right-hand expectation:

- **TD target** (a 1-step sample of the expectation):
  $$
  R_{t+1} + \gamma\, V(S_{t+1})
  $$
- **TD error** (how far off our current estimate is):
  $$
  \delta_t \;=\; R_{t+1} + \gamma\, V(S_{t+1}) - V(S_t)
  $$
- **Update rule** – take a small step toward the target:

$$
\boxed{\; V(S_t) \;\leftarrow\; V(S_t) \;+\; \alpha\, \underbrace{\bigl[\, R_{t+1} + \gamma\, V(S_{t+1}) - V(S_t)\,\bigr]}_{\delta_t} \;}
$$

This algorithm is called **TD(0)**:
- **T**emporal **D**ifference – we learn from the difference between what we expected, $V(S_t)$, and what we just saw, $R_{t+1} + \gamma V(S_{t+1})$.
- The **0** denotes *one-step lookahead*; later we'll meet n-step TD and TD(λ).

It's also called **bootstrapping**: the target uses our own current estimate $V(S_{t+1})$ — we improve an estimate using another estimate.

### MC vs. TD(0) at a glance

|  | Monte Carlo | TD(0) |
|---|---|---|
| When do we update? | After the episode ends | After every step |
| Target | Full return $G_t$ (sampled) | $R_{t+1} + \gamma V(S_{t+1})$ (sampled + bootstrapped) |
| Bias / variance | Unbiased, high variance | Biased (uses current $V$), lower variance |
| Episodes required | Must terminate | Works step-by-step, even in continuing tasks |

### 🧪 §4.1 What we'll do

Fix a strategy (`π_1` or `π_2`), run many episodes, and apply the TD(0) update after each step. Knobs:
- `alpha` (learning rate) – how large each update step is.
- `gamma` (discount) – same meaning as before.


In [ ]:
# ▶ Tip: keep π automatic for the default 200k episodes; use "manual" only for a handful.
episodes: int = 200000

π = π_1  # Callable or "manual"
# π = π_2
# π = "manual"

env = make_frozen_lake(show_values=True)

################################################################################
v = np.zeros((env.observation_space.n,))
gamma = 0.9  # discount factor γ
alpha = 0.9  # step size α (a big α converges fast on this deterministic world)
################################################################################

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            if π == "manual":
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)
            new_state, reward, terminated, truncated, info_dict = env.step(action)

            # ╭──────────────────────────────────────────────────────────╮
            # │ ★ CORE IDEA — TD(0) update                               │
            # │   target = r + γ · V(s')       (0 if terminal, no boot.) │
            # │   δ      = target − V(s)                    (TD error)   │
            # │   V(s)  ← V(s) + α · δ                      (step toward │
            # │                                              the target) │
            # ╰──────────────────────────────────────────────────────────╯
            done = terminated or truncated
            td_target = reward + (0.0 if done else gamma * v[new_state])
            td_error = td_target - v[state]
            v[state] += alpha * td_error

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

In [ ]:
import matplotlib.pyplot as plt

# Assume `v` already contains your learned value function (shape: n_states)

# 1. Create an env in rgb_array mode
env = make_frozen_lake(render_mode="rgb_array", show_values=True)

# 2. Reset and set the value function for rendering
state, _ = env.reset()
env.unwrapped.set_v(v)
env.unwrapped.set_episode("final")
env.unwrapped.set_info({"Note": "Final value function V^π(s)"})

# 3. Render one frame as an RGB array
rgb = env.render()  # shape: (H, W, 3), dtype=uint8

env.close()

# 4. Plot with matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.axis("off")
plt.title("Final value function $V^π(s)$ using TD(0)")
plt.show()

### 🧠 What did we learn from V(s)?

- For a fixed strategy π, we can estimate how “good” each state is on average.
- Monte Carlo and TD(0) give us two different ways to do this:
  - Monte Carlo: learn from complete episodes.
  - TD(0): learn step by step.
- But V(s) alone does **not** tell us which action is best in each state.

## 🌊 Bonus: Slippery Frozen Lake (Stochastic Environment)

So far the world was **deterministic**: choose "right" → move right.

Now we turn on **slippery ice**:
- With probability `success_rate` (here 3/4) the intended action happens.
- Otherwise the elf slips to one of the perpendicular directions.
- So the **same action can lead to different next states** – the transition $p(s' \mid s, a)$ is now non-trivially stochastic.

Try running TD(0) on the slippery lake and observe:
- Does the path become less predictable?
- How do the learned values $V(s)$ differ from the deterministic case?
- Tiles next to holes become more dangerous – you can see it in their value.


In [ ]:
episodes: int = 50000
π = π_1  # try π_2 as well — it is much less robust on slippery ice

# ── ★ The ONLY change from §4: slippery=True ──
env = make_frozen_lake(show_values=True, slippery=True)

v = np.zeros((env.observation_space.n,))
gamma = 0.9
alpha = 0.05  # smaller α because TD targets are noisier in the slippery world

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not (terminated or truncated):
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            action = act(state=state, π=π)
            new_state, reward, terminated, truncated, _ = env.step(action)

            # ── ★ Same TD(0) update as §4 — the algorithm didn't change ──
            done = terminated or truncated
            td_target = reward + (0.0 if done else gamma * v[new_state])
            v[state] += alpha * (td_target - v[state])
            # ─────────────────────────────────────────────────────────────

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
                "slippery": True,
            })
            env.render()
            state = new_state

        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(200)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

In [ ]:
import matplotlib.pyplot as plt

# Render the final learned V^π(s) on the slippery lake
env = make_frozen_lake(render_mode="rgb_array", show_values=True, slippery=True)
state, _ = env.reset()
env.unwrapped.set_v(v)
env.unwrapped.set_episode("final")
env.unwrapped.set_info({"Note": "Final V^π(s) on the slippery lake"})

rgb = env.render()
env.close()

plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.axis("off")
plt.title("Final value function $V^\\pi(s)$ — slippery world (TD(0))")
plt.show()

## 🚀 Next: From Evaluating a Strategy to Learning a Good One

So far we:
- Fixed a strategy π.
- Asked: “How good is each state if we follow π?”

Next, we’d like to:
- **Improve** the strategy.
- Choose actions that lead to higher long‑term rewards.

For that, it’s more convenient to learn a value for **state–action pairs**, Q(s, a), instead of just V(s).

## 🔀 $V(s)$ vs $Q(s, a)$ — Two Ways to Score Decisions

We've been working with **state values** $V^\pi(s)$. Let's now meet their close cousin, **action values** $Q^\pi(s, a)$, and see why both exist.

### 🧭 Intuition first — a chess analogy

- **$V(s)$** = "How good is this position?" — *one number per board state.*
- **$Q(s, a)$** = "How good is this position **if I play this specific move**?" — *one number per (board, move) pair.*

A grandmaster doesn't just say "this position is +0.5"; they compare candidate moves. That comparison is exactly what $Q$ captures.

### 📐 Side-by-side definitions

|  | $V^\pi(s)$ — *state value* | $Q^\pi(s, a)$ — *action value* |
|---|---|---|
| **Question it answers** | "I'm in state $s$ and follow $\pi$. How much reward do I expect?" | "I'm in state $s$, take action $a$ **once**, then follow $\pi$. How much reward do I expect?" |
| **Input** | just the state | state **and** action |
| **Table size** | $\lvert S \rvert$ entries | $\lvert S \rvert \cdot \lvert A \rvert$ entries |
| **Definition** | $\mathbb{E}_\pi[\, G_t \mid S_t = s\,]$ | $\mathbb{E}_\pi[\, G_t \mid S_t = s,\, A_t = a\,]$ |

### 🔗 How $V$ and $Q$ are related

$V$ is the action-weighted average of $Q$ under $\pi$:

$$
V^\pi(s) \;=\; \sum_a \pi(a \mid s)\, Q^\pi(s, a)
$$

And for the **optimal** policy, $V^*$ is simply the largest $Q^*$ in each state:

$$
V^*(s) \;=\; \max_a Q^*(s, a)
$$

So $V$ is a *summary* of $Q$; $Q$ is $V$ *broken down by action*.

### 🧊 A Frozen Lake moment

Imagine the elf is one tile away from a hole and the value map says $V(s) = 0.4$ — a decent state.

- **With only $V$:** the elf knows "this state is ok", but *which direction* is safe? Without the transition model $p(s' \mid s, a)$ (which neighbour does each action lead to?), it simply can't tell.
- **With $Q$:** the elf reads $Q(s, \leftarrow)$, $Q(s, \downarrow)$, $Q(s, \rightarrow)$, $Q(s, \uparrow)$ and picks the largest. Done — no model needed.

### 🧠 Why $Q$ matters for *control* (learning a good strategy)

Acting well means answering: *"What action should I take in state $s$?"* There are two routes:

1. **Model-based (use $V$):** for each candidate action, simulate where you'd land and look at $V(s')$.  Requires knowing the transition model $p(s' \mid s, a)$.
2. **Model-free (use $Q$):** just read off $\arg\max_a Q(s, a)$. **No model needed.**

In most real problems we **don't have the transition model**, so $Q$ wins.

### ⚖️ When to use each

| Use $V^\pi$ when… | Use $Q^\pi$ when… |
|---|---|
| You only need to **evaluate** a fixed policy | You want to **pick actions** / learn a good policy |
| The transition model is known (planning, dynamic programming) | The model is unknown (model-free RL) |
| You prefer a small table ($\lvert S \rvert$ entries) | You can afford $\lvert A \rvert\times$ as much memory |
| Policy/action-selection is handled separately (e.g. actor-critic) | You want to derive the policy *directly* from values |

> 💡 **Rule of thumb for this workshop:**  
> *Evaluating* a fixed strategy → $V$ is enough.  
> *Improving* a strategy from experience → you'll almost always want $Q$.

In §5 below, we learn $Q$ directly from experience with **Q-learning**.


## 🎯 Q-Learning: Learning $Q(s, a)$ from Experience

Now that we know what $Q$ is, let's learn it **from experience**, just like we did for $V$ — but now we keep one value per (state, action) pair instead of one per state.

### From $Q$ to a strategy: ε-greedy

Given a Q-table, the most natural strategy is: *in each state, pick the action with the highest $Q$-value.* That's **greedy**. But a purely greedy agent never tries anything new and might miss a better path it never explored. So we perturb it:

> **ε-greedy** — with probability $\varepsilon$ act *uniformly at random*, otherwise act greedily.

High $\varepsilon$ = lots of exploration. Low $\varepsilon$ = lots of exploitation. In practice we **anneal** $\varepsilon$ from high to low over training (our `EpsilonGreedyPolicy` does exactly that via `decay()`).

### The Q-learning update (off-policy)

After observing a transition $(s, a, r, s')$, we update:

$$
\boxed{\; Q(s, a) \;\leftarrow\; Q(s, a) \;+\; \alpha\,\Big[\, r + \gamma\, \max_{a'} Q(s', a') \;-\; Q(s, a) \,\Big] \;}
$$

The key subtlety is the $\max_{a'}$: the target bootstraps from the **greedy** next action, even though our behaviour policy acts ε-greedy. This is what makes Q-learning **off-policy** — we learn about the greedy policy while behaving more exploratory.

> 🔄 For contrast, **SARSA** (on-policy) plugs in $Q(s', a_{\text{next}})$, using the action that our behaviour policy actually takes next.

In the next cell we implement this update and watch $Q(s, a)$ grow on the slippery grid.


## §5. Q-Learning with a Policy $\pi_q$ (ε-greedy from Q)

In [ ]:
episodes = 15000

env = make_frozen_lake(show_values=True, slippery=True)
env.unwrapped.set_show_q_labels(True)  # show per-action Q-values on the grid

nS = env.observation_space.n
nA = env.action_space.n

q = np.zeros((nS, nA))  # Q-table — will be mutated IN PLACE every step
alpha = 0.1
gamma = 0.9

# ╭──────────────────────────────────────────────────────────────╮
# │ ★ Behaviour policy: ε-greedy DERIVED FROM q.                 │
# │   EpsilonGreedyPolicy stores `q` by reference — so every     │
# │   Q-learning update below immediately changes what π does.   │
# ╰──────────────────────────────────────────────────────────────╯
π = EpsilonGreedyPolicy(q=q, epsilon=0.5, epsilon_decay=0.0001)
# π = π_1   # swap in a fixed policy to see pure Q-evaluation
# π = π_2

try:
    for i in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False

        while not (terminated or truncated):
            env.unwrapped.set_q(q)
            env.unwrapped.set_episode(i)

            # Behaviour step: act ε-greedy according to current q
            action = act(state=state, π=π)

            new_state, reward, terminated, truncated, _ = env.step(action)

            # ╭──────────────────────────────────────────────────────────╮
            # │ ★ CORE IDEA — Q-learning update (OFF-POLICY)             │
            # │   target = r + γ · max_{a'} Q(s', a')                    │
            # │   δ      = target − Q(s, a)                              │
            # │   Q(s,a) ← Q(s, a) + α · δ                               │
            # │ The max bootstraps from the GREEDY policy even though    │
            # │ we acted ε-greedy — that's why this is "off-policy".     │
            # ╰──────────────────────────────────────────────────────────╯
            td_target = reward + gamma * np.max(q[new_state, :])
            td_error = td_target - q[state, action]
            q[state, action] += alpha * td_error

            env.unwrapped.set_info({
                "Policy": type(π).__name__,
                "Epsilon": f"{getattr(π, 'epsilon', 'N/A'):.3f}"
                if hasattr(π, "epsilon")
                else "N/A",
                "Reward": reward,
            })
            env.render()

            state = new_state

        # Decay epsilon once per episode (only has effect for EpsilonGreedyPolicy)
        if hasattr(π, "decay"):
            π.decay()

except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

In [ ]:
import matplotlib.pyplot as plt

# 1. Create an env in rgb_array mode
env = make_frozen_lake(render_mode="rgb_array", show_values=True)

# 2. Reset and set the Q-values for rendering
state, _ = env.reset()
env.unwrapped.set_q(q)
env.unwrapped.set_episode("final")
env.unwrapped.set_info({"Note": "Final Q-values Q(s, a)"})

# 3. Render one frame as an RGB array
rgb = env.render()  # shape: (H, W, 3), dtype=uint8

env.close()

# 4. Plot with matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.axis("off")
plt.title("Final action-value function $Q(s, a)$ learned via Q-learning")
plt.show()

### What to observe

- At the beginning, the agent explores a lot (ε is high).
- Over time, ε decreases and the agent uses the learned Q(s, a) more.
- The Q-values on the map show which actions are preferred in each state.
- Even in the slippery world, the agent can learn a strategy that reliably reaches the goal.

## ✅ Summary

In this notebook you:

1. **Played** Frozen Lake yourself to build intuition for states, actions, and rewards.
2. Defined two simple **policies** (`π_1`, `π_2`) as functions $\pi(a \mid s) \to [0, 1]$.
3. Learned to estimate the **value function** $V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$ under a fixed policy using two approaches:
    - **Monte Carlo rollouts** – average the return $G_t$ over many complete episodes (first-visit).
    - **TD(0)** – derived from the Bellman equation $V^\pi(s) = \mathbb{E}_\pi[R_{t+1} + \gamma V^\pi(S_{t+1})]$; update after every single step using the TD error $\delta_t$.
4. Explored the **slippery** (stochastic) version of the environment and saw how the same policy produces different – and lower-valued – trajectories.
5. Moved from *evaluating* a fixed strategy to *learning* a good one with **Q-learning**, using $Q(s, a)$ plus an **ε-greedy** policy to balance exploration and exploitation.

### Where to go next

- Change the strategy $\pi$ and watch how $V^\pi(s)$ changes.
- Compare MC and TD(0) on the *same* policy: which stabilizes faster? Which is noisier?
- Try the slippery world with π₂ (pure Right/Down). Why does it fail more often?
- Tweak `gamma` (0.5 vs 0.99) and `alpha` (0.01 vs 0.9). What happens?
- Implement **SARSA** (on-policy) and compare against Q-learning on the slippery lake.

These ideas are the foundation of many modern RL algorithms, including the deep-RL methods we'll see in the next lab.
